# 🧠 Hallucination Sentinel — Colab GPU Runner
**Pipeline:** TruthfulQA → OPT-6.7B (4-bit) → EigenScore + LN-Entropy + Lexical Similarity → AUROC

> ⚡ Make sure your runtime is set to **GPU**: `Runtime → Change runtime type → A100 / T4`

## ✅ Step 1 — Verify GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected! Go to Runtime → Change runtime type → GPU")

## 📦 Step 2 — Install Dependencies
Colab already has PyTorch with CUDA. We only install the missing packages.

In [ ]:
%%capture
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.29.0 \
    bitsandbytes>=0.43.0 \
    sentence-transformers>=2.7.0 \
    datasets>=2.19.0 \
    rouge-score>=0.1.2 \
    scikit-learn>=1.4.0 \
    fastapi pydantic

print("✅ Dependencies installed")

## 📁 Step 3 — Mount Google Drive & Clone / Upload Project

**Option A (recommended):** Mount Google Drive where your project zip is stored.  
**Option B:** Clone from GitHub (if your repo is on GitHub).

In [ ]:
# ── Option A: Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Update this path to where you uploaded the zip on your Drive:
DRIVE_ZIP = '/content/drive/MyDrive/hallucination-detection.zip'

import os, zipfile
PROJECT_DIR = '/content/hallucination-detection'

if not os.path.exists(PROJECT_DIR):
    print("Extracting project...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall('/content/')
    print("✅ Extracted to", PROJECT_DIR)
else:
    print("✅ Project already extracted")

In [ ]:
# ── Option B: GitHub (uncomment and fill in your repo URL) ────────────────────
# !git clone https://github.com/YOUR_USERNAME/hallucination-detection.git /content/hallucination-detection
# PROJECT_DIR = '/content/hallucination-detection'

In [ ]:
# Verify project structure
import os
PROJECT_DIR = '/content/hallucination-detection'
os.chdir(PROJECT_DIR)
!ls -la

## ⚙️ Step 4 — Patch Model Loader for Colab GPU
On Colab A100 (40 GB VRAM), we don't need CPU offload. This patch makes the model run fully on GPU for maximum speed.

In [ ]:
import torch

# Detect available VRAM and set memory limits accordingly
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Detected VRAM: {vram_gb:.1f} GB")

# Write an optimised llm_loader for Colab
loader_code = '''
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_model(model_name="facebook/opt-6.7b"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    if vram_gb >= 20:
        # A100 / High-VRAM: run fully on GPU
        print(f"[INFO] Large GPU detected ({vram_gb:.0f} GB VRAM) — loading model fully on GPU")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="cuda:0",
        )
    elif vram_gb >= 8:
        # T4 / V100: fit on GPU with small CPU overflow
        print(f"[INFO] Medium GPU detected ({vram_gb:.0f} GB VRAM) — loading with minimal CPU offload")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            max_memory={0: f"{int(vram_gb*0.9)}GiB", "cpu": "8GiB"},
        )
    else:
        # Fallback: heavy CPU offload (slow)
        print(f"[INFO] Small GPU ({vram_gb:.0f} GB VRAM) — using CPU offload")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            max_memory={0: "5GiB", "cpu": "20GiB"},
        )

    return tokenizer, model
'''

with open('models/llm_loader.py', 'w') as f:
    f.write(loader_code)

print("✅ llm_loader.py patched for Colab GPU")

## 🚀 Step 5 — Run the Pipeline

Start with `--limit 50` to test, then remove the limit for the full 817 questions.

In [ ]:
# Quick test run: 50 questions (~15-20 min on A100, ~45 min on T4)
!python pipeline/run_dataset.py --limit 50

In [ ]:
# Full run: all 817 TruthfulQA questions (~3-4 hrs on A100, ~8-10 hrs on T4)
# NOTE: Resume is automatic — if the cell is interrupted, just re-run it!
# !python pipeline/run_dataset.py

## 📊 Step 6 — Evaluate Results (AUROC)

In [ ]:
!python evaluate_tqa.py

## 💾 Step 7 — Save Results Back to Google Drive

In [ ]:
import shutil, os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
SAVE_DIR = f'/content/drive/MyDrive/hallucination-results-{timestamp}'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy results
for fname in ['data/tfulresults.csv', 'data/tfulraw_data.jsonl']:
    if os.path.exists(fname):
        shutil.copy(fname, SAVE_DIR)
        print(f"✅ Saved {fname} → {SAVE_DIR}")
    else:
        print(f"⚠️  {fname} not found — pipeline may not have run yet")

print(f"\n📁 All results saved to: {SAVE_DIR}")

## 📈 Step 8 — Quick Visualisation (optional)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

df = pd.read_csv('data/tfulresults.csv').dropna(subset=['label'])
df['label'] = df['label'].astype(int)

print(f"Total: {len(df)} | Factual: {(df.label==0).sum()} | Hallucination: {(df.label==1).sum()}")

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Hallucination Sentinel — ROC Curves (TruthfulQA)', fontsize=14, fontweight='bold')

metrics_cfg = {
    'eigenscore':         {'flip': False, 'color': '#6366f1'},
    'ln_entropy':         {'flip': False, 'color': '#f59e0b'},
    'avg_token_prob':     {'flip': True,  'color': '#10b981'},
    'lexical_similarity': {'flip': True,  'color': '#ef4444'},
    'length_mean':        {'flip': False, 'color': '#8b5cf6'},
    'length_std':         {'flip': False, 'color': '#06b6d4'},
}

for ax, (metric, cfg) in zip(axes.flatten(), metrics_cfg.items()):
    if metric not in df.columns:
        ax.set_title(f'{metric} (missing)')
        continue
    scores = (-df[metric] if cfg['flip'] else df[metric]).fillna(0)
    fpr, tpr, _ = roc_curve(df['label'], scores)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=cfg['color'], lw=2, label=f'AUC = {roc_auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(loc='lower right')
    ax.set_facecolor('#f8fafc')

plt.tight_layout()
plot_path = 'data/roc_curves.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"✅ Plot saved to {plot_path}")

---
### ℹ️ Tips
- **Runtime disconnects?** Your data is safe in `/content/hallucination-detection/data/` until the runtime is fully reset. The pipeline **auto-resumes** from where it left off.
- **Save frequently** using Step 7 to avoid losing results on a runtime timeout.
- **Colab Pro background execution:** `Runtime → Run all` then close the browser tab — the notebook keeps running.
- **Check GPU memory:** `!nvidia-smi` in any cell.